# 05b — Preprocessing Fenotipik

**Tujuan.** Menyiapkan fitur fenotipik (tanpa citra) untuk baseline tabular, dengan mencegah leakage:
- pembersihan deterministik (global, tercatat);
- keputusan missing value berbasis audit 05a;
- pipeline sklearn di `src/pheno_preprocessing.py` yang **di-fit hanya pada train tiap fold**;
- demonstrasi fit/transform pada satu fold `fold_strat` dan satu situs LOSO. Tidak ada model yang dilatih.

| CELL | Isi |
|---|---|
| 1 | CONFIG |
| 2 | Muat data, hash awal, label, kandidat fitur |
| 3 | Pembersihan deterministik |
| 3c | Penanda integritas citra (`image_integrity_flag`) |
| 3d | Simpan `pheno_clean.csv`, `cleaning_log.csv`, `drop_log.csv` |
| 4 | Keputusan missing value dan fitur terpilih |
| 5 | Pipeline sklearn (import dari `src/`) |
| 6 | Demonstrasi: fold `fold_strat` |
| 7 | Demonstrasi: LOSO |
| 8 | Audit akhir dan `pheno_verification_summary.md` |

**Input (read-only):** `dataset/cleaned_dataset/cleaned_master_dataset.csv`, `data/folds.csv` (skema v2: kolom `split` dev/test), `data/exclusions.csv`, `data/audit_flags.csv`, `data/audit_report.md`.

Subjek di `exclusions.csv` tidak diproses dan ikut dicatat di `drop_log.csv`. Subjek **test** ikut dibersihkan secara deterministik (tanpa fit), tetapi **tidak pernah** dipakai di demonstrasi fit (train maupun validation).

**Output:** `data/pheno_clean.csv`, `data/cleaning_log.csv`, `data/drop_log.csv`, `data/missing_decision.csv`, `data/pheno_verification_summary.md`, `src/pheno_preprocessing.py` (modul pipeline).

### CELL 1 — CONFIG

**Tujuan:** semua path dan parameter di satu tempat. **Output:** variabel global, versi library tercetak.

Ambang di bawah adalah pilihan metodologis yang ditulis di `missing_decision.csv`, bukan standar baku:
- `HANDEDNESS_MAX_MISSING_PCT` dan `MIN_CATEGORY_N` hanya memicu "perlu ditinjau", **tidak** membuang fitur secara otomatis.
- `CAT_MISSING_STRATEGY = "most_frequent"`: missing handedness diisi modus train. Kategori `Missing` eksplisit tidak dipakai karena missing handedness terkonsentrasi di beberapa situs, sehingga kategori itu bisa menjadi proksi situs.

In [ ]:
import hashlib
import importlib
import re
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.exceptions import NotFittedError
from sklearn.utils.validation import check_is_fitted
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# ---------- Path (relatif terhadap root proyek) ----------
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "dataset").exists() and (PROJECT_ROOT.parent / "dataset").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent             # dijalankan dari notebook/
DATA_DIR = PROJECT_ROOT / "data"
SRC_DIR = PROJECT_ROOT / "src"

CLEANED_PATH = PROJECT_ROOT / "dataset" / "cleaned_dataset" / "cleaned_master_dataset.csv"
FOLDS_PATH = DATA_DIR / "folds.csv"
AUDIT_FLAGS_PATH = DATA_DIR / "audit_flags.csv"
AUDIT_REPORT_PATH = DATA_DIR / "audit_report.md"
EXCLUSIONS_PATH = DATA_DIR / "exclusions.csv"

PHENO_CLEAN_PATH = DATA_DIR / "pheno_clean.csv"
CLEANING_LOG_PATH = DATA_DIR / "cleaning_log.csv"
DROP_LOG_PATH = DATA_DIR / "drop_log.csv"
MISSING_DECISION_PATH = DATA_DIR / "missing_decision.csv"
SUMMARY_PATH = DATA_DIR / "pheno_verification_summary.md"
PIPELINE_MODULE_PATH = SRC_DIR / "pheno_preprocessing.py"

# ---------- Seed ----------
SEED = 42
np.random.seed(SEED)

# ---------- Fitur ----------
FEATURE_CANDIDATES = ["AGE_AT_SCAN", "SEX", "FIQ", "HANDEDNESS_CATEGORY"]
OPTIONAL_FEATURES = ["FIQ_missing"]               # hanya masuk jika keputusan audit mendukung
DX_TO_LABEL = {1: 1, 0: 0}                        # DX_GROUP (recode Notebook 04) -> label
LABEL_NAMES = {1: "ASD", 0: "TD"}

# ---------- Pembersihan ----------
MISSING_CODES = {-9999, -9999.0, "-9999", "-9999.0"}
AGE_VALID_RANGE = (5, 70)                         # sama dengan Notebook 04/05a
FIQ_VALID_RANGE = (40, 160)
SEX_VALID = {1, 2}
HANDEDNESS_VALID = {"R", "L", "Ambi"}
DROP_ON_AUDIT_FLAGS = ["flag_id_mismatch"]        # bukti konkret data tidak valid (ID salah pasangan)

# ---------- Keputusan missing ----------
ALPHA = 0.05
SITE_MISSING_STRUCTURAL = 50.0                    # % NaN per situs
HANDEDNESS_MAX_MISSING_PCT = 5.0
MIN_CATEGORY_N = 20
CAT_MISSING_STRATEGY = "most_frequent"            # "most_frequent" | "missing_category"

# ---------- Demonstrasi ----------
DEMO_FOLD_SCHEME = "fold_strat"
DEMO_FOLD = 0
DEMO_LOSO_SITE = None                             # None = situs eligible pertama (urut nama) tanpa konflik

# ---------- Kolom log ----------
CLEANING_LOG_COLUMNS = ["step", "SUB_ID", "column", "old_value", "new_value", "action", "reason"]
DROP_LOG_COLUMNS = ["SUB_ID", "SITE_ID", "step", "reason", "evidence"]

assert SEED == 42 and CAT_MISSING_STRATEGY in {"most_frequent", "missing_category"}
assert (PROJECT_ROOT / "dataset").exists(), f"Root proyek tidak ditemukan dari {Path.cwd()}"
DATA_DIR.mkdir(parents=True, exist_ok=True)
SRC_DIR.mkdir(parents=True, exist_ok=True)
print("PROJECT_ROOT:", PROJECT_ROOT)
print(f"python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__} | scikit-learn {sklearn.__version__}")

### CELL 2 — Muat data, hash awal, label, kandidat fitur

**Tujuan:** memastikan semua input 05a ada (berhenti jika tidak), mencatat hash awal, mencocokkan master dan `folds.csv` lewat `SUB_ID`, mengambil `y = label` dari `folds.csv`, memeriksa konsistensi `DX_GROUP × label`, membaca hasil verifikasi silang dari `audit_report.md`, dan menetapkan kandidat fitur.

**Output:** `master`, `folds`, `aflags`, `audit_text`, `HASH_MASTER_BEFORE`, `HASH_FOLDS_BEFORE`.

In [ ]:
def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


REVIEW = []   # semua hal "perlu ditinjau" -> summary


def needs_review(text):
    REVIEW.append(text)
    print("[PERLU DITINJAU]", text)


missing_inputs = [p for p in (CLEANED_PATH, FOLDS_PATH, EXCLUSIONS_PATH, AUDIT_FLAGS_PATH, AUDIT_REPORT_PATH) if not p.exists()]
if missing_inputs:
    raise FileNotFoundError("Input tidak ditemukan: " + ", ".join(str(p.relative_to(PROJECT_ROOT)) for p in missing_inputs)
                            + ". Jalankan notebook/05a_audit_folds.ipynb sampai selesai terlebih dahulu.")

HASH_MASTER_BEFORE = sha256_file(CLEANED_PATH)
HASH_FOLDS_BEFORE = sha256_file(FOLDS_PATH)
print("SHA-256 cleaned_master_dataset.csv (awal):", HASH_MASTER_BEFORE)
print("SHA-256 folds.csv (awal)                 :", HASH_FOLDS_BEFORE)

master_all = pd.read_csv(CLEANED_PATH)
folds = pd.read_csv(FOLDS_PATH)
exclusions = pd.read_csv(EXCLUSIONS_PATH)
aflags_all = pd.read_csv(AUDIT_FLAGS_PATH, keep_default_na=False, na_values=[""])
audit_text = AUDIT_REPORT_PATH.read_text(encoding="utf-8")
assert "split" in folds.columns and set(folds["split"]) == {"dev", "test"}, "folds.csv bukan skema v2 (dev/test); jalankan 05a"
EXCL_IDS = set(exclusions["SUB_ID"].astype(int))
assert EXCL_IDS <= set(master_all["SUB_ID"]) and not EXCL_IDS & set(folds["SUB_ID"]), "exclusions.csv tidak konsisten dengan folds.csv"
master = master_all[~master_all["SUB_ID"].isin(EXCL_IDS)].reset_index(drop=True)
aflags = aflags_all[aflags_all["SUB_ID"].isin(folds["SUB_ID"])].reset_index(drop=True)
print(f"Master {len(master_all)} → {len(master)} setelah eksklusi {sorted(EXCL_IDS)} | dev {int((folds['split'] == 'dev').sum())} | "
      f"test {int((folds['split'] == 'test').sum())}")

FOLD_COLS = [c for c in folds.columns if c == "fold_strat" or c.startswith("repeated_fold_")]
SPLIT_COLS = FOLD_COLS + ["loso_site"]
need_folds = {"SUB_ID", "SITE_ID", "label", "group_id", "split", "loso_site", "loso_group_conflict",
              "strata_fallback", "loso_eligible", "fold_strat"}
assert need_folds <= set(folds.columns), f"Kolom folds.csv hilang: {need_folds - set(folds.columns)}"
assert {"SUB_ID", "DX_GROUP", *FEATURE_CANDIDATES, *OPTIONAL_FEATURES} <= set(master.columns)
assert {"SUB_ID", "group_id", "smri_voxel_sha256"} <= set(aflags.columns)

# Pencocokan SUB_ID
for name, frame in [("master", master), ("folds", folds), ("audit_flags", aflags)]:
    assert frame["SUB_ID"].notna().all() and frame["SUB_ID"].is_unique, f"SUB_ID kosong/ganda di {name}"
only_master = set(master["SUB_ID"]) - set(folds["SUB_ID"])
only_folds = set(folds["SUB_ID"]) - set(master["SUB_ID"])
assert not only_master and not only_folds, f"SUB_ID tidak cocok: hanya master={sorted(only_master)[:10]}, hanya folds={sorted(only_folds)[:10]}"
assert set(aflags["SUB_ID"]) == set(folds["SUB_ID"]), "SUB_ID audit_flags.csv tidak sama dengan folds.csv"

# Target dari folds.csv dan konsistensi dengan DX_GROUP
base = master[["SUB_ID", "DX_GROUP"]].merge(folds[["SUB_ID", "label"]], on="SUB_ID", how="inner", validate="one_to_one")
dx_label_ct = pd.crosstab(base["DX_GROUP"], base["label"])
print("Crosstab DX_GROUP × label:")
display(dx_label_ct)
assert set(base["DX_GROUP"].unique()) == set(DX_TO_LABEL), f"Nilai DX_GROUP tak terduga: {sorted(base['DX_GROUP'].unique())}"
assert (base["DX_GROUP"].map(DX_TO_LABEL) == base["label"]).all(), "DX_GROUP dan label tidak konsisten"
assert (dx_label_ct.astype(bool).sum(axis=1) == 1).all() and (dx_label_ct.astype(bool).sum(axis=0) == 1).all(), "Pemetaan bukan one-to-one"

# Verifikasi silang ke data mentah: dibaca dari audit_report.md (bukan dihitung ulang)
m = re.search(r"Verifikasi silang DX_GROUP[^\n]*?:\s*(LOLOS|GAGAL)\s*\(([^)]*)\)", audit_text)
DX_RAW_CHECK = f"{m.group(1)} ({m.group(2)})" if m else "perlu ditinjau"
print("Verifikasi silang DX_GROUP vs data mentah (audit_report.md):", DX_RAW_CHECK)
if not m:
    needs_review("Hasil verifikasi silang DX_GROUP vs data mentah tidak ditemukan di audit_report.md.")
elif m.group(1) != "LOLOS":
    needs_review(f"Verifikasi silang DX_GROUP di 05a: {DX_RAW_CHECK}.")

# Kandidat fitur (bukan seluruh kolom numerik)
FORBIDDEN_IN_MASTER = [c for c in master.columns if c not in FEATURE_CANDIDATES + OPTIONAL_FEATURES]
print("\nKolom master yang tidak dipakai sebagai fitur:", FORBIDDEN_IN_MASTER)
FEATURE_REASON = {
    "AGE_AT_SCAN": "demografi dasar; bukan instrumen diagnostik",
    "SEX": "demografi dasar; bukan instrumen diagnostik",
    "FIQ": "kemampuan kognitif umum; bukan instrumen diagnostik ASD (konfounder, lihat audit)",
    "HANDEDNESS_CATEGORY": "kandidat; masuk hanya jika keputusan missing/kategori di CELL 4 mendukung",
    "FIQ_missing": "opsional; masuk hanya jika audit tidak menunjukkan risiko proksi SITE_ID",
}
y_all = folds.set_index("SUB_ID")["label"]
print("\nTarget: label (dari folds.csv), 1 = ASD, 0 = TD")
print("Distribusi label:", y_all.map(LABEL_NAMES).value_counts().to_dict())
print("Kandidat fitur:")
for f in FEATURE_CANDIDATES + OPTIONAL_FEATURES:
    print(f"  - {f}: {FEATURE_REASON[f]}")

### CELL 2b — Verifikasi CELL 2

In [ ]:
assert len(master) == len(folds) == len(aflags)
assert not set(FEATURE_CANDIDATES + OPTIONAL_FEATURES) & {"DX_GROUP", "label", "SUB_ID", "SITE_ID", "group_id"}
assert y_all.isin([0, 1]).all()
print(f"Baris master={len(master)} | folds={len(folds)} | audit_flags={len(aflags)}")
print(f"Skema split: {SPLIT_COLS}")
print("Distribusi kelas per situs:")
display(pd.crosstab(folds["SITE_ID"], folds["label"].map(LABEL_NAMES), margins=True, margins_name="TOTAL"))

### CELL 3 — Pembersihan deterministik

**Tujuan:** memeriksa tipe data, kategori, nilai tidak valid, satuan, dan kode missing pada kandidat fitur. Kode `-9999` diubah ke NaN hanya jika ditemukan, dan setiap perubahan dicatat. **Tanpa** imputasi, scaling, encoding, atau penghapusan baris karena missing. Baris hanya dikeluarkan bila ada bukti konkret di audit (`DROP_ON_AUDIT_FLAGS`).

**Output:** `pheno` (SUB_ID + kandidat fitur), `cleaning_records`, `drop_records`.

In [ ]:
cleaning_records, drop_records = [], []


def log_clean(step, column, action, reason, sub_ids=None, old=None, new=None):
    if sub_ids is None:
        cleaning_records.append({"step": step, "SUB_ID": "", "column": column, "old_value": "",
                                 "new_value": "", "action": action, "reason": reason})
        return
    for i, sid in enumerate(sub_ids):
        cleaning_records.append({"step": step, "SUB_ID": int(sid), "column": column,
                                 "old_value": None if old is None else old[i],
                                 "new_value": None if new is None else new[i], "action": action, "reason": reason})


pheno = master[["SUB_ID"] + FEATURE_CANDIDATES + OPTIONAL_FEATURES].copy()

# 1) Kode missing (-9999)
for col in FEATURE_CANDIDATES + OPTIONAL_FEATURES:
    s = pheno[col]
    mask = s.map(lambda v: (not pd.isna(v)) and (v in MISSING_CODES or str(v).strip() in {str(c) for c in MISSING_CODES}))
    if mask.any():
        log_clean("3.1", col, "MODIFIED", "kode missing -9999 -> NaN", pheno.loc[mask, "SUB_ID"].tolist(),
                  s[mask].tolist(), [np.nan] * int(mask.sum()))
        pheno.loc[mask, col] = np.nan
    else:
        log_clean("3.1", col, "NO_CHANGE", "tidak ada kode -9999")
    print(f"{col:20s} kode -9999: {int(mask.sum())}")

# 2) Spasi / ejaan kategori
for col in ["HANDEDNESS_CATEGORY"]:
    s = pheno[col]
    stripped = s.map(lambda v: v.strip() if isinstance(v, str) else v)
    changed = s.notna() & (stripped != s)
    if changed.any():
        log_clean("3.2", col, "MODIFIED", "hapus spasi di tepi", pheno.loc[changed, "SUB_ID"].tolist(),
                  s[changed].tolist(), stripped[changed].tolist())
        pheno[col] = stripped
    else:
        log_clean("3.2", col, "NO_CHANGE", "tidak ada spasi berlebih")

# 3) Tipe data numerik: nilai non-numerik dilaporkan, tidak diubah diam-diam
for col in ["AGE_AT_SCAN", "FIQ", "SEX", "FIQ_missing"]:
    coerced = pd.to_numeric(pheno[col], errors="coerce")
    bad = pheno[col].notna() & coerced.isna()
    if bad.any():
        needs_review(f"{col}: {int(bad.sum())} nilai non-numerik {pheno.loc[bad, col].unique()[:5].tolist()}")
        log_clean("3.3", col, "FLAGGED", "nilai non-numerik, tidak diubah", pheno.loc[bad, "SUB_ID"].tolist())
    else:
        pheno[col] = coerced
        log_clean("3.3", col, "NO_CHANGE", f"tipe numerik valid ({pheno[col].dtype})")

# 4) Rentang, kategori, satuan
checks = []
for col, (lo, hi) in [("AGE_AT_SCAN", AGE_VALID_RANGE), ("FIQ", FIQ_VALID_RANGE)]:
    out = pheno[col].notna() & ~pheno[col].between(lo, hi)
    checks.append({"kolom": col, "cek": f"rentang [{lo}, {hi}]", "n_tidak_valid": int(out.sum())})
sex_bad = pheno["SEX"].notna() & ~pheno["SEX"].isin(SEX_VALID)
hand_bad = pheno["HANDEDNESS_CATEGORY"].notna() & ~pheno["HANDEDNESS_CATEGORY"].isin(HANDEDNESS_VALID)
fiqm_bad = ~pheno["FIQ_missing"].isin([0, 1])
checks += [{"kolom": "SEX", "cek": f"kode {sorted(SEX_VALID)}", "n_tidak_valid": int(sex_bad.sum())},
           {"kolom": "HANDEDNESS_CATEGORY", "cek": f"kategori {sorted(HANDEDNESS_VALID)}", "n_tidak_valid": int(hand_bad.sum())},
           {"kolom": "FIQ_missing", "cek": "biner 0/1", "n_tidak_valid": int(fiqm_bad.sum())}]
fiqm_incons = pheno["FIQ_missing"].astype(int).ne(pheno["FIQ"].isna().astype(int))
checks.append({"kolom": "FIQ_missing", "cek": "== FIQ.isna()", "n_tidak_valid": int(fiqm_incons.sum())})
age_unit = "tahun" if pheno["AGE_AT_SCAN"].max() < 100 else "bukan tahun?"
checks.append({"kolom": "AGE_AT_SCAN", "cek": f"satuan terdeteksi: {age_unit}", "n_tidak_valid": int(age_unit != "tahun")})
check_tbl = pd.DataFrame(checks)
display(check_tbl)
for r in check_tbl.itertuples():
    if r.n_tidak_valid:
        needs_review(f"{r.kolom}: {r.n_tidak_valid} nilai gagal cek '{r.cek}'. Tidak diubah otomatis.")
        log_clean("3.4", r.kolom, "FLAGGED", f"gagal cek {r.cek}; tidak diubah")
    else:
        log_clean("3.4", r.kolom, "NO_CHANGE", f"lolos cek {r.cek}")

# 5) Baris tidak valid berdasarkan bukti konkret dari audit
for flag in DROP_ON_AUDIT_FLAGS:
    if flag not in aflags.columns:
        needs_review(f"Kolom {flag} tidak ada di audit_flags.csv; cek baris tidak valid dilewati.")
        continue
    bad_ids = aflags.loc[aflags[flag].astype(str).str.lower().eq("true"), "SUB_ID"].tolist()
    site = folds.set_index("SUB_ID")["SITE_ID"]
    for sid in bad_ids:
        drop_records.append({"SUB_ID": int(sid), "SITE_ID": site[sid], "step": "3.5",
                             "reason": "ID subjek tidak cocok dengan file citra", "evidence": f"audit_flags.{flag}=True"})
dropped_ids = {r["SUB_ID"] for r in drop_records}
pheno = pheno[~pheno["SUB_ID"].isin(dropped_ids)].reset_index(drop=True)
print(f"\nBaris dikeluarkan (bukti konkret): {len(dropped_ids)}")
print(f"Perubahan nilai: {sum(r['action'] == 'MODIFIED' for r in cleaning_records)} baris log")

### CELL 3b — Verifikasi CELL 3

In [ ]:
for col in FEATURE_CANDIDATES + OPTIONAL_FEATURES:
    assert not pheno[col].isin([-9999, -9999.0]).any(), f"{col}: masih ada -9999"
    assert not pheno[col].astype(str).str.strip().isin({"-9999", "-9999.0"}).any()
assert len(pheno) == len(master) - len(dropped_ids)
assert pheno["SUB_ID"].is_unique
assert check_tbl["n_tidak_valid"].sum() == 0, "Ada nilai tidak valid (lihat 'perlu ditinjau')"
modified = [r for r in cleaning_records if r["action"] == "MODIFIED"]
print(f"Baris: {len(pheno)} | kolom: {list(pheno.columns)}")
print("Perubahan:", "tidak ada perubahan" if not modified else f"{len(modified)} nilai diubah")
print("NaN per kolom:", pheno.isna().sum().to_dict())
print("Rentang:", {c: (pheno[c].min(), pheno[c].max()) for c in ["AGE_AT_SCAN", "FIQ"]})
print("Kategori SEX:", pheno["SEX"].value_counts().to_dict(), "| HANDEDNESS:",
      pheno["HANDEDNESS_CATEGORY"].value_counts(dropna=False).to_dict())

### CELL 3c — Penanda integritas citra (`image_integrity_flag`)

**Tujuan:** membaca kelompok citra identik dari `audit_flags.csv` (`group_id`, `smri_voxel_sha256`) **tanpa menghitung ulang hash**. `image_integrity_flag = True` untuk subjek yang berada di kelompok citra identik **dan** fenotipiknya berbeda. Fenotipik tetap diproses, tidak ada yang dihapus, dan tidak ada yang ditulis ke `drop_log.csv`. Eksklusi untuk model citra diputuskan di 05d.

**Output:** kolom `image_integrity_flag` di `pheno`.

In [ ]:
img = aflags[["SUB_ID", "group_id", "smri_voxel_sha256"]].merge(
    folds[["SUB_ID", "group_id"]].rename(columns={"group_id": "group_id_folds"}), on="SUB_ID", validate="one_to_one")
assert (img["group_id"] == img["group_id_folds"]).all(), "group_id audit_flags.csv dan folds.csv berbeda"
img["group_size"] = img.groupby("group_id")["SUB_ID"].transform("size")
multi = img[img["group_size"] > 1].merge(pheno, on="SUB_ID", how="left")
pheno_cols = FEATURE_CANDIDATES
if len(multi):
    differs = (multi[pheno_cols].astype(str).groupby(multi["group_id"]).nunique() > 1).any(axis=1)
    multi["fenotipik_berbeda"] = multi["group_id"].map(differs)
    multi["voxel_hash_sama"] = multi.groupby("group_id")["smri_voxel_sha256"].transform("nunique").eq(1)
    print("Kelompok citra identik (dari audit_flags.csv):")
    display(multi[["group_id", "SUB_ID"] + pheno_cols + ["voxel_hash_sama", "fenotipik_berbeda"]])
    assert multi["voxel_hash_sama"].all(), "Anggota satu group_id punya hash voxel berbeda; periksa 05a"
    flag_ids = set(multi.loc[multi["fenotipik_berbeda"], "SUB_ID"])
else:
    print("Tidak ada kelompok citra identik di audit_flags.csv.")
    flag_ids = set()
pheno["image_integrity_flag"] = pheno["SUB_ID"].isin(flag_ids)
for sid in sorted(flag_ids):
    log_clean("3c", "image_integrity_flag", "FLAGGED",
              "citra identik dengan SUB_ID lain, fenotipik berbeda; eksklusi model citra diputuskan di 05d", [sid])
if flag_ids:
    needs_review(f"image_integrity_flag=True untuk SUB_ID {sorted(flag_ids)}: citra identik dengan fenotipik berbeda. "
                 "Fenotipik tetap dipakai di 05b; keputusan untuk model citra di 05d.")

### CELL 3d — Simpan `pheno_clean.csv`, `cleaning_log.csv`, `drop_log.csv` + verifikasi

**Output:** tiga file di `data/`. `drop_log.csv` tetap dibuat (dengan header) meskipun kosong.

In [ ]:
PHENO_CLEAN_COLUMNS = ["SUB_ID"] + FEATURE_CANDIDATES + OPTIONAL_FEATURES + ["image_integrity_flag"]
pheno = pheno[PHENO_CLEAN_COLUMNS].sort_values("SUB_ID").reset_index(drop=True)
pheno.to_csv(PHENO_CLEAN_PATH, index=False)
cleaning_log = pd.DataFrame(cleaning_records, columns=CLEANING_LOG_COLUMNS)
cleaning_log.to_csv(CLEANING_LOG_PATH, index=False)
drop_log = pd.concat([exclusions[DROP_LOG_COLUMNS], pd.DataFrame(drop_records, columns=DROP_LOG_COLUMNS)],
                     ignore_index=True)
drop_log.to_csv(DROP_LOG_PATH, index=False)

chk = pd.read_csv(PHENO_CLEAN_PATH)
assert list(chk.columns) == PHENO_CLEAN_COLUMNS and len(chk) == len(pheno)
assert not {"DX_GROUP", "label", "SITE_ID", "group_id", "smri_path"} & set(chk.columns)
assert list(pd.read_csv(DROP_LOG_PATH).columns) == DROP_LOG_COLUMNS
assert set(pd.read_csv(DROP_LOG_PATH)["SUB_ID"]) == EXCL_IDS | dropped_ids
assert not chk["SUB_ID"].isin(EXCL_IDS).any()
assert set(chk["SUB_ID"]) == set(folds["SUB_ID"]) - dropped_ids
print(f"pheno_clean.csv : {chk.shape} | image_integrity_flag=True: {int(chk['image_integrity_flag'].sum())}")
print(f"cleaning_log.csv: {len(cleaning_log)} baris | aksi: {cleaning_log['action'].value_counts().to_dict()}")
print(f"drop_log.csv    : {len(drop_log)} baris (eksklusi 05a: {len(exclusions)}, drop 05b: {len(drop_records)})")
print("NaN per kolom   :", chk.isna().sum()[lambda s: s > 0].to_dict())

### CELL 4 — Keputusan missing value dan fitur terpilih

**Tujuan:** menyusun `missing_decision.csv` per fitur: jumlah dan persen missing (dihitung dari `pheno`), pola per situs dan label (**dibaca dari audit 05a**: `audit_flags.csv` dan tabel "Uji asosiasi missing" / "Missing per situs" di `audit_report.md`), keputusan, dan alasan. Jika hasil audit tidak terbaca, keputusannya "perlu ditinjau".

Aturan:
- FIQ: dipakai, imputasi median dari train tiap fold.
- FIQ_missing: tidak dipakai di eksperimen awal jika missing FIQ terkonsentrasi di situs tertentu atau berbeda antar label (risiko proksi SITE_ID).
- HANDEDNESS_CATEGORY: dipakai hanya jika missing kecil, tidak ada kategori langka, dan missing tidak berasosiasi dengan label.
- AGE_AT_SCAN, SEX: dipakai jika lengkap dan valid.
- Tidak ada fitur yang dibuang hanya karena ambang persen, dan tidak ada subjek yang dibuang karena missing.

**Output:** `missing_decision.csv`, `SELECTED_FEATURES`.

In [ ]:
def read_md_table(text, title):
    """Baca tabel markdown di bawah judul '### <title>' pada audit_report.md. None jika tidak ada."""
    m = re.search(rf"^### {re.escape(title)}[ \t]*\n\s*\n((?:\|.*\|[ \t]*\n?)+)", text, re.M)
    if not m:
        return None
    rows = [[c.strip() for c in line.strip().strip("|").split("|")] for line in m.group(1).strip().splitlines()]
    return pd.DataFrame(rows[2:], columns=rows[0])


def audit_p(assoc, test_name):
    if assoc is None or "uji" not in assoc.columns or "p" not in assoc.columns:
        return None
    row = assoc.loc[assoc["uji"] == test_name, "p"]
    try:
        return float(row.iloc[0]) if len(row) else None
    except ValueError:
        return None


assoc = read_md_table(audit_text, "Uji asosiasi missing")
site_miss_tbl = read_md_table(audit_text, "Missing per situs (situs dengan NaN)")
p_fiq_site, p_fiq_label = audit_p(assoc, "FIQ NaN × SITE_ID"), audit_p(assoc, "FIQ NaN × label")
p_hand_site, p_hand_label = audit_p(assoc, "HANDEDNESS NaN × SITE_ID"), audit_p(assoc, "HANDEDNESS NaN × label")
print("p dari audit_report.md:", {"FIQ×situs": p_fiq_site, "FIQ×label": p_fiq_label,
                                  "HAND×situs": p_hand_site, "HAND×label": p_hand_label})
if assoc is None:
    needs_review("Tabel 'Uji asosiasi missing' tidak ditemukan di audit_report.md.")

# Pola per situs/label dari audit_flags.csv (fiq_nan, handedness_nan)
af = aflags.merge(folds[["SUB_ID", "label"]], on="SUB_ID")
af = af[~af["SUB_ID"].isin(dropped_ids)]


def as_bool(s):
    return s.astype(str).str.lower().eq("true")


def pattern(flag_col):
    if flag_col not in af.columns:
        return None, None, None
    f = as_bool(af[flag_col])
    by_site = af.assign(f=f).groupby("SITE_ID")["f"].agg(["sum", "size"])
    by_site["pct"] = by_site["sum"] / by_site["size"] * 100
    by_site = by_site[by_site["sum"] > 0].sort_values("pct", ascending=False)
    site_txt = "; ".join(f"{s}: {int(r['sum'])}/{int(r['size'])} ({r['pct']:.1f}%)" for s, r in by_site.iterrows()) or "-"
    by_lab = af.assign(f=f).groupby("label")["f"].agg(["sum", "size"])
    lab_txt = "; ".join(f"{LABEL_NAMES[l]}: {int(r['sum'])}/{int(r['size'])} ({r['sum'] / r['size'] * 100:.1f}%)"
                        for l, r in by_lab.iterrows())
    return f, by_site, (site_txt, lab_txt)


fiq_flag, fiq_sites, fiq_txt = pattern("fiq_nan")
hand_flag, hand_sites, hand_txt = pattern("handedness_nan")
if fiq_flag is not None:
    assert (fiq_flag.values == pheno.set_index("SUB_ID").loc[af["SUB_ID"], "FIQ"].isna().values).all(), \
        "fiq_nan di audit_flags.csv tidak cocok dengan FIQ.isna(); jalankan ulang 05a"
if hand_flag is not None:
    assert (hand_flag.values == pheno.set_index("SUB_ID").loc[af["SUB_ID"], "HANDEDNESS_CATEGORY"].isna().values).all()

decisions = []


def add_decision(feature, decision, reason, site_txt="-", label_txt="-", source="-"):
    n = int(pheno[feature].isna().sum())
    decisions.append({"fitur": feature, "n_missing": n, "pct_missing": round(n / len(pheno) * 100, 2),
                      "pola_situs": site_txt, "pola_label": label_txt, "keputusan": decision,
                      "alasan": reason, "sumber_audit": source})


# AGE_AT_SCAN, SEX
for col in ["AGE_AT_SCAN", "SEX"]:
    ok = pheno[col].notna().all() and check_tbl.loc[check_tbl["kolom"] == col, "n_tidak_valid"].sum() == 0
    add_decision(col, "dipakai" if ok else "perlu ditinjau",
                 "lengkap (0 NaN) dan lolos cek validitas" if ok else "ada NaN atau nilai tidak valid",
                 source="pheno_clean (CELL 3)")

# FIQ
max_site_pct = float(fiq_sites["pct"].max()) if fiq_sites is not None and len(fiq_sites) else 0.0
struct_sites = fiq_sites.index[fiq_sites["pct"] >= SITE_MISSING_STRUCTURAL].tolist() if fiq_sites is not None else []
fiq_reason = (f"missing {pheno['FIQ'].isna().mean():.1%}; imputasi median di-fit pada train tiap fold. "
              f"Missing terkonsentrasi di {struct_sites} (struktural per situs): subjek situs ini mendapat median train, "
              "sehingga FIQ mereka tidak informatif. Audit 05a menandai FIQ sebagai konfounder; laporkan ablasi dengan/tanpa FIQ.")
add_decision("FIQ", "dipakai (imputasi median train per fold)", fiq_reason,
             *(fiq_txt or ("perlu ditinjau", "perlu ditinjau")),
             source=f"audit_flags.fiq_nan; audit_report p(situs)={p_fiq_site}, p(label)={p_fiq_label}")

# Tindak lanjut audit 05a terkait FIQ
m_viq = re.search(r"(\d+) dari (\d+) subjek FIQ NaN memiliki VIQ dan PIQ valid", audit_text)
viq_txt = f"{m_viq.group(1)} dari {m_viq.group(2)}" if m_viq else "jumlah perlu dicek di audit_report.md"
needs_review(f"VIQ/PIQ untuk subjek dengan FIQ kosong ({viq_txt}, audit 05a): tidak dipakai di eksperimen awal; "
             "opsional sebagai analisis sensitivitas.")
needs_review(f"Imputasi median membuat FIQ konstan untuk {struct_sites or 'situs dengan missing struktural'}, sehingga "
             "bisa menjadi penanda situs pada model pohon. Rencana: ablasi dengan dan tanpa FIQ, dan perhatikan hasil "
             "LOSO pada situs tersebut.")

# FIQ_missing
if p_fiq_site is None or p_fiq_label is None or fiq_flag is None:
    fm_dec, fm_reason = "perlu ditinjau", "hasil uji asosiasi missing FIQ tidak terbaca dari audit"
elif (p_fiq_site < ALPHA and max_site_pct >= SITE_MISSING_STRUCTURAL) or p_fiq_label < ALPHA:
    fm_dec = "tidak dipakai di eksperimen awal"
    fm_reason = (f"missing FIQ terkonsentrasi di situs {struct_sites} (p situs={p_fiq_site:.2e}, maks {max_site_pct:.0f}%/situs); "
                 f"p label={p_fiq_label:.3f}. FIQ_missing ≈ indikator situs → risiko proksi SITE_ID. Boleh diuji sebagai ablasi terpisah.")
else:
    fm_dec = "opsional (ablasi)"
    fm_reason = f"tidak ada konsentrasi situs/label yang nyata (p situs={p_fiq_site:.2e}, p label={p_fiq_label:.3f})"
add_decision("FIQ_missing", fm_dec, fm_reason, *(fiq_txt or ("-", "-")), source="audit_flags.fiq_nan; audit_report")
if fm_dec == "perlu ditinjau":
    needs_review("Keputusan FIQ_missing: " + fm_reason)

# HANDEDNESS_CATEGORY
hand_counts = pheno["HANDEDNESS_CATEGORY"].value_counts()
rare = hand_counts[hand_counts < MIN_CATEGORY_N].to_dict()
hand_pct = pheno["HANDEDNESS_CATEGORY"].isna().mean() * 100
dist_txt = ", ".join(f"{k}={v}" for k, v in hand_counts.items())
if p_hand_label is None or hand_flag is None:
    hd_dec, hd_reason = "perlu ditinjau", "hasil uji asosiasi missing handedness tidak terbaca dari audit"
elif rare:
    hd_dec, hd_reason = "perlu ditinjau", f"kategori langka (< {MIN_CATEGORY_N}): {rare}; distribusi {dist_txt}"
elif hand_pct > HANDEDNESS_MAX_MISSING_PCT:
    hd_dec, hd_reason = "perlu ditinjau", f"missing {hand_pct:.1f}% > {HANDEDNESS_MAX_MISSING_PCT}%; tinjau pola sebelum dipakai"
elif p_hand_label < ALPHA:
    hd_dec, hd_reason = "perlu ditinjau", f"missing berasosiasi dengan label (p={p_hand_label:.3f})"
else:
    hd_dec = f"dipakai (imputasi {CAT_MISSING_STRATEGY} train per fold, one-hot)"
    hd_reason = (f"missing {hand_pct:.1f}%, tidak berasosiasi dengan label (p={p_hand_label:.3f}); distribusi {dist_txt}, "
                 f"tidak ada kategori < {MIN_CATEGORY_N}. Missing terkonsentrasi di beberapa situs (p situs={p_hand_site:.2e}), "
                 "jadi tidak dibuat kategori 'Missing' eksplisit (bisa menjadi proksi situs).")
add_decision("HANDEDNESS_CATEGORY", hd_dec, hd_reason, *(hand_txt or ("-", "-")),
             source="audit_flags.handedness_nan; audit_report")
if hd_dec == "perlu ditinjau":
    needs_review("Keputusan HANDEDNESS_CATEGORY: " + hd_reason)

missing_decision = pd.DataFrame(decisions)
missing_decision.to_csv(MISSING_DECISION_PATH, index=False)
display(missing_decision[["fitur", "n_missing", "pct_missing", "keputusan"]])
for r in missing_decision.itertuples():
    print(f"\n[{r.fitur}] {r.keputusan}\n  alasan : {r.alasan}\n  situs  : {r.pola_situs}\n  label  : {r.pola_label}")

SELECTED_FEATURES = [r.fitur for r in missing_decision.itertuples() if r.keputusan.startswith("dipakai")]
print("\nFITUR TERPILIH:", SELECTED_FEATURES)
print("Tidak dipakai / perlu ditinjau:", [f for f in missing_decision["fitur"] if f not in SELECTED_FEATURES])

### CELL 4b — Verifikasi CELL 4

In [ ]:
md_chk = pd.read_csv(MISSING_DECISION_PATH)
assert set(md_chk["fitur"]) == set(FEATURE_CANDIDATES + OPTIONAL_FEATURES)
assert md_chk["keputusan"].notna().all() and md_chk["alasan"].notna().all()
assert set(SELECTED_FEATURES) <= set(FEATURE_CANDIDATES + OPTIONAL_FEATURES)
assert len(pheno) == len(master) - len(dropped_ids), "Ada subjek yang dibuang karena missing"
print(f"missing_decision.csv: {md_chk.shape} | fitur terpilih: {len(SELECTED_FEATURES)} -> {SELECTED_FEATURES}")
print("Total NaN pada fitur terpilih (sebelum imputasi per fold):", pheno[SELECTED_FEATURES].isna().sum().to_dict())

### CELL 5 — Pipeline sklearn (`src/pheno_preprocessing.py`)

**Tujuan:** mengimpor modul pipeline dan membangun dua pipeline **yang belum di-fit**:
- `make_pipeline_tree`: median numerik; imputasi kategori lalu one-hot; tanpa scaling.
- `make_pipeline_linear_nn`: median + `StandardScaler` numerik; imputasi kategori lalu `OneHotEncoder(handle_unknown="ignore")`.

Hanya `SELECTED_FEATURES` yang masuk. Kategori tidak di-encode ordinal.

**Output:** `pp` (modul), `X_all`, `data` (pheno + kolom split), `y`.

In [ ]:
assert PIPELINE_MODULE_PATH.exists(), f"{PIPELINE_MODULE_PATH} tidak ada"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
import pheno_preprocessing as pp
pp = importlib.reload(pp)

data = pheno.merge(folds, on="SUB_ID", how="inner", validate="one_to_one").sort_values("SUB_ID").reset_index(drop=True)
X_all = pp.select_features(data, {"features": SELECTED_FEATURES})
y = data["label"].astype(int)
pp.assert_no_forbidden(X_all.columns)
num_f, cat_f, bin_f = pp.split_feature_types(X_all.columns)
print("X_all:", X_all.shape, "| numerik:", num_f, "| kategori:", cat_f, "| biner:", bin_f)
print("Kolom terlarang yang terdeteksi di data (tidak masuk X):", pp.forbidden_columns(data.columns))
print(pp.make_pipeline_tree(list(X_all.columns), CAT_MISSING_STRATEGY))
print(pp.make_pipeline_linear_nn(list(X_all.columns), CAT_MISSING_STRATEGY))

### CELL 5b — Verifikasi CELL 5

In [ ]:
assert list(X_all.columns) == [f for f in pp.ALLOWED_FEATURES if f in SELECTED_FEATURES]
assert "label" not in X_all.columns and y.isin([0, 1]).all() and len(X_all) == len(y) == len(data)
for maker in (pp.make_pipeline_tree, pp.make_pipeline_linear_nn):
    try:
        check_is_fitted(maker(list(X_all.columns)).named_steps["prep"])
        raise AssertionError("Pipeline seharusnya belum di-fit")
    except NotFittedError:
        pass
print(f"X_all {X_all.shape} | y {y.shape} | label: {y.map(LABEL_NAMES).value_counts().to_dict()}")
print("Pipeline belum di-fit: OK (fit hanya di CELL 6/7 pada train)")

### CELL 6 — Demonstrasi: fold `fold_strat`

**Tujuan:** menunjukkan urutan yang benar: **pisahkan train/val dulu → fit pada X train → transform keduanya**. Hanya subjek **dev**: fold `DEMO_FOLD` menjadi validation, fold dev lainnya train. Subjek test tidak dipakai. Dicetak: jumlah subjek, distribusi label, shape sebelum/sesudah, NaN, nama fitur akhir. Assert numerik: `imputer.statistics_` = median X train; `scaler.mean_` dan `scale_` = mean dan std (ddof=0) X train **setelah imputasi median train** (karena scaler menerima data yang sudah diimputasi).

Interpretasi: setelah scaling, **train** bermean ≈0 dan std ≈1. **Validation tidak harus** 0/1 karena memakai parameter train; selisihnya wajar.

**Output:** `DEMO_RESULTS["fold_strat"]`.

In [ ]:
DEMO_RESULTS = {}
IS_DEV = data["split"].eq("dev").values
IS_TEST = data["split"].eq("test").values
print(f"Demo hanya pada dev: {int(IS_DEV.sum())} subjek | test disisihkan: {int(IS_TEST.sum())}")


def run_demo(name, train_mask, val_mask):
    train_mask, val_mask = np.asarray(train_mask), np.asarray(val_mask)
    assert not (train_mask & val_mask).any() and ((train_mask | val_mask) == IS_DEV).all(), "train+val harus = dev"
    assert not (IS_TEST & (train_mask | val_mask)).any(), "subjek test masuk train/val"
    g_tr, g_val = set(data.loc[train_mask, "group_id"]), set(data.loc[val_mask, "group_id"])
    assert not g_tr & g_val, f"group_id muncul di train dan val: {sorted(g_tr & g_val)[:5]}"

    # 1) Split dulu
    X_tr, X_val = X_all[train_mask], X_all[val_mask]
    y_tr, y_val = y[train_mask], y[val_mask]
    print(f"=== {name} ===")
    print(f"train: n={len(X_tr)} {y_tr.map(LABEL_NAMES).value_counts().to_dict()} | "
          f"val: n={len(X_val)} {y_val.map(LABEL_NAMES).value_counts().to_dict()}")
    print(f"NaN sebelum transform -> train {X_tr.isna().sum().to_dict()} | val {X_val.isna().sum().to_dict()}")

    res = {"n_train": len(X_tr), "n_val": len(X_val), "shape_before": (X_tr.shape, X_val.shape)}
    # 2) Fit pada train saja, 3) transform keduanya
    for kind, maker in [("tree", pp.make_pipeline_tree), ("linear_nn", pp.make_pipeline_linear_nn)]:
        pipe = maker(list(X_all.columns), CAT_MISSING_STRATEGY).fit(X_tr)
        Z_tr, Z_val = pipe.transform(X_tr), pipe.transform(X_val)
        names = list(pipe.named_steps["prep"].get_feature_names_out())
        n_nan = int(np.isnan(Z_tr).sum() + np.isnan(Z_val).sum())
        print(f"[{kind}] shape {X_tr.shape}/{X_val.shape} -> {Z_tr.shape}/{Z_val.shape} | NaN sesudah: {n_nan}")
        print(f"[{kind}] fitur akhir ({len(names)}): {names}")
        assert n_nan == 0 and Z_tr.shape[1] == Z_val.shape[1] == len(names)
        res[kind] = {"pipe": pipe, "Z_tr": Z_tr, "Z_val": Z_val, "names": names, "n_nan": n_nan}

    # 4) Assert numerik: parameter = statistik train
    checks = {}
    if num_f:
        med_tr, med_all, med_val = X_tr[num_f].median(), X_all[num_f].median(), X_val[num_f].median()
        for kind in ("tree", "linear_nn"):
            stat = pp.get_numeric_step(res[kind]["pipe"], "imputer").statistics_
            assert np.allclose(stat, med_tr.values), f"{kind}: imputer.statistics_ != median train"
        print("\nMedian numerik  train (dipakai):", med_tr.round(3).to_dict())
        print("Median numerik  seluruh data   :", med_all.round(3).to_dict(), "(TIDAK dipakai)")
        print("Median numerik  validation     :", med_val.round(3).to_dict(), "(TIDAK dipakai)")
        checks["imputer_eq_train_median"] = True

        scaler = pp.get_numeric_step(res["linear_nn"]["pipe"], "scaler")
        X_tr_imp = X_tr[num_f].fillna(med_tr)
        assert np.allclose(scaler.mean_, X_tr_imp.mean().values), "scaler.mean_ != mean train"
        assert np.allclose(scaler.scale_, X_tr_imp.std(ddof=0).values), "scaler.scale_ != std(ddof=0) train"
        assert scaler.n_samples_seen_ == len(X_tr), "scaler tidak di-fit pada seluruh (dan hanya) train"
        checks["scaler_eq_train_stats"] = True

        names_lin = res["linear_nn"]["names"]
        idx = [names_lin.index(c) for c in num_f]
        X_val_imp = X_val[num_f].fillna(med_tr)
        tbl = pd.DataFrame({
            "train_mean_sebelum": X_tr_imp.mean(), "train_std_sebelum": X_tr_imp.std(ddof=0),
            "val_mean_sebelum": X_val_imp.mean(), "val_std_sebelum": X_val_imp.std(ddof=0),
            "train_mean_sesudah": res["linear_nn"]["Z_tr"][:, idx].mean(axis=0),
            "train_std_sesudah": res["linear_nn"]["Z_tr"][:, idx].std(axis=0),
            "val_mean_sesudah": res["linear_nn"]["Z_val"][:, idx].mean(axis=0),
            "val_std_sesudah": res["linear_nn"]["Z_val"][:, idx].std(axis=0)}, index=num_f).round(4)
        print("\nMean/std sebelum (setelah imputasi median train) dan sesudah scaling (pipeline linear_nn):")
        display(tbl)
        assert np.allclose(tbl["train_mean_sesudah"], 0, atol=1e-6) and np.allclose(tbl["train_std_sesudah"], 1, atol=1e-6)
        print("Train: mean≈0, std≈1 (sesuai definisi). Validation memakai mean/std train, jadi tidak harus 0/1.")
        res["scaling_table"] = tbl

    if cat_f:
        cat_imp = res["linear_nn"]["pipe"].named_steps["prep"].named_transformers_["cat"].named_steps["imputer"]
        mode_tr = pp._as_category_str(X_tr[cat_f]).mode().iloc[0].tolist()
        if CAT_MISSING_STRATEGY == "most_frequent":
            assert list(cat_imp.statistics_) == mode_tr, f"imputer kategori != modus train: {cat_imp.statistics_} vs {mode_tr}"
        print(f"Imputasi kategori ({CAT_MISSING_STRATEGY}) dari train:", dict(zip(cat_f, cat_imp.statistics_)))
        checks["cat_imputer_eq_train_mode"] = CAT_MISSING_STRATEGY == "most_frequent"
    res["checks"] = checks
    return res


fold_mask_val = IS_DEV & (data[DEMO_FOLD_SCHEME] == DEMO_FOLD).values
DEMO_RESULTS[f"{DEMO_FOLD_SCHEME}={DEMO_FOLD}"] = run_demo(f"{DEMO_FOLD_SCHEME}: fold {DEMO_FOLD} = validation (dev)",
                                                            IS_DEV & ~fold_mask_val, fold_mask_val)

### CELL 6b — Verifikasi CELL 6

In [ ]:
r = DEMO_RESULTS[f"{DEMO_FOLD_SCHEME}={DEMO_FOLD}"]
assert r["n_train"] + r["n_val"] == int(IS_DEV.sum())
assert all(r["checks"].values())
assert r["tree"]["n_nan"] == 0 and r["linear_nn"]["n_nan"] == 0
print(f"Fold {DEMO_FOLD}: train={r['n_train']} val={r['n_val']} | tree {r['tree']['Z_tr'].shape[1]} fitur | "
      f"linear_nn {r['linear_nn']['Z_tr'].shape[1]} fitur | cek: {r['checks']}")

### CELL 7 — Demonstrasi: LOSO

**Tujuan:** sama dengan CELL 6, tetapi validation = seluruh subjek **dev** satu situs `loso_eligible` (`DEMO_LOSO_SITE`; jika `None`, situs eligible pertama tanpa konflik LOSO). Train = subjek dev situs lain. Test tidak dipakai.

**Output:** `DEMO_RESULTS["loso=<situs>"]`.

In [ ]:
eligible_sites = sorted(data.loc[data["loso_eligible"].astype(bool) & ~data["loso_group_conflict"].astype(bool), "loso_site"].unique())
loso_site = DEMO_LOSO_SITE or eligible_sites[0]
assert loso_site in eligible_sites, f"{loso_site} tidak loso_eligible atau punya konflik LOSO"
loso_mask_val = IS_DEV & (data["loso_site"] == loso_site).values
print(f"Situs LOSO eligible (dev): {len(eligible_sites)} | demo: {loso_site}")
DEMO_RESULTS[f"loso={loso_site}"] = run_demo(f"LOSO: situs {loso_site} = validation (dev)", IS_DEV & ~loso_mask_val, loso_mask_val)

### CELL 7b — Verifikasi CELL 7

In [ ]:
r = DEMO_RESULTS[f"loso={loso_site}"]
assert r["n_val"] == int((data["loso_site"] == loso_site).sum()) and r["n_train"] + r["n_val"] == int(IS_DEV.sum())
assert all(r["checks"].values()) and r["linear_nn"]["n_nan"] == 0
assert set(data.loc[loso_mask_val, "SITE_ID"]) <= {loso_site} | set(data.loc[data["loso_group_conflict"].astype(bool), "SITE_ID"])
print(f"LOSO {loso_site}: train={r['n_train']} val={r['n_val']} | cek: {r['checks']}")

### CELL 8 — Audit akhir dan `pheno_verification_summary.md`

**Tujuan:** assert akhir (kolom terlarang, target, ID, NaN setelah transform, fitur tercatat, fit hanya pada train, hash input tidak berubah) dan pemeriksaan kebocoran citra identik **tanpa menghitung ulang hash**: setiap `group_id` dan setiap `smri_voxel_sha256` hanya boleh berada di satu fold pada setiap skema. Jika dilanggar, notebook berhenti dan perbaikan dilakukan di 05a.

**Output:** `pheno_verification_summary.md`.

In [ ]:
# 1) Fitur dan target
pp.assert_no_forbidden(X_all.columns)
assert "label" not in X_all.columns and "DX_GROUP" not in X_all.columns
assert data["SUB_ID"].is_unique and set(data["SUB_ID"]) == set(folds["SUB_ID"]) - dropped_ids
assert set(pd.read_csv(PHENO_CLEAN_PATH)["SUB_ID"]) == set(data["SUB_ID"])

# 2) Transform dan fit
for name, r in DEMO_RESULTS.items():
    assert r["tree"]["n_nan"] == 0 and r["linear_nn"]["n_nan"] == 0, f"{name}: NaN setelah transform"
    assert all(r["checks"].values()), f"{name}: parameter tidak sama dengan statistik train"
    if num_f:
        assert pp.get_numeric_step(r["linear_nn"]["pipe"], "scaler").n_samples_seen_ == r["n_train"]
FINAL_FEATURES = {k: DEMO_RESULTS[next(iter(DEMO_RESULTS))][k]["names"] for k in ("tree", "linear_nn")}
for name, r in DEMO_RESULTS.items():
    for k in ("tree", "linear_nn"):
        assert r[k]["names"] == FINAL_FEATURES[k], f"{name}/{k}: nama fitur berbeda antar split"

# 3) Kebocoran citra identik (dibaca dari folds.csv + audit_flags.csv)
leak = folds.merge(aflags[["SUB_ID", "smri_voxel_sha256"]], on="SUB_ID", validate="one_to_one")
violations = []
for col in SPLIT_COLS + ["split"]:
    n_g = leak.groupby("group_id")[col].nunique()
    violations += [f"{col}: group_id {g} di {n} fold" for g, n in n_g[n_g > 1].items()]
    vh = leak[leak["smri_voxel_sha256"].fillna("").ne("")]
    n_h = vh.groupby("smri_voxel_sha256")[col].nunique()
    violations += [f"{col}: hash voxel {h[:12]} di {n} fold" for h, n in n_h[n_h > 1].items()]
if violations:
    raise RuntimeError("KEBOCORAN citra identik antar-fold:\n  " + "\n  ".join(violations[:20])
                       + "\nPerbaiki di notebook 05a (CELL 3c group_id dan CELL 7 folds), lalu jalankan ulang 05a dan 05b.")
n_multi_groups = int((leak.groupby("group_id").size() > 1).sum())
print(f"Kebocoran citra identik: tidak ada ({len(SPLIT_COLS)} skema, {n_multi_groups} group_id berisi >1 subjek)")

# 4) Hash input
HASH_MASTER_AFTER = sha256_file(CLEANED_PATH)
HASH_FOLDS_AFTER = sha256_file(FOLDS_PATH)
print("SHA-256 master awal/akhir:", HASH_MASTER_BEFORE, "/", HASH_MASTER_AFTER)
print("SHA-256 folds  awal/akhir:", HASH_FOLDS_BEFORE, "/", HASH_FOLDS_AFTER)
assert HASH_MASTER_BEFORE == HASH_MASTER_AFTER, "cleaned_master_dataset.csv berubah!"
assert HASH_FOLDS_BEFORE == HASH_FOLDS_AFTER, "folds.csv berubah!"

# 5) Ringkasan markdown
def md_table(frame):
    frame = frame.reset_index() if not isinstance(frame.index, pd.RangeIndex) else frame
    cols = [str(c) for c in frame.columns]
    out = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    out += ["| " + " | ".join(str(v).replace("|", "/") for v in row) + " |" for row in frame.itertuples(index=False)]
    return "\n".join(out)


lines = ["# Pheno Verification Summary — Notebook 05b", "",
         f"- Sumber: `{CLEANED_PATH.relative_to(PROJECT_ROOT).as_posix()}` (SHA-256 `{HASH_MASTER_AFTER}`, tidak berubah)",
         f"- Folds: `{FOLDS_PATH.relative_to(PROJECT_ROOT).as_posix()}` (SHA-256 `{HASH_FOLDS_AFTER}`, tidak berubah)",
         f"- scikit-learn {sklearn.__version__}, pandas {pd.__version__}, seed {SEED}",
         f"- Subjek: {len(data)} (ASD {int(y.sum())}, TD {int((y == 0).sum())}); dev {int(IS_DEV.sum())}, test {int(IS_TEST.sum())}",
         f"- Eksklusi 05a: {sorted(EXCL_IDS)} (exclusions.csv); dikeluarkan 05b: {len(dropped_ids)}",
         "- Demonstrasi fit hanya pada dev; subjek test tidak dipakai untuk fit maupun validation",
         f"- Verifikasi silang DX_GROUP vs data mentah (dari audit_report.md): {DX_RAW_CHECK}",
         f"- DX_GROUP × label: one-to-one (1→1 ASD, 0→0 TD)", "",
         "## Fitur", "",
         f"- Terpilih: {SELECTED_FEATURES}",
         f"- Fitur akhir pipeline tree ({len(FINAL_FEATURES['tree'])}): {FINAL_FEATURES['tree']}",
         f"- Fitur akhir pipeline linear/NN ({len(FINAL_FEATURES['linear_nn'])}): {FINAL_FEATURES['linear_nn']}",
         f"- Imputasi kategori: {CAT_MISSING_STRATEGY}; numerik: median train per fold", "",
         "## Keputusan missing value", "", md_table(missing_decision[["fitur", "n_missing", "pct_missing", "keputusan", "alasan"]]), "",
         "## Pembersihan deterministik", "",
         f"- Nilai diubah: {sum(r['action'] == 'MODIFIED' for r in cleaning_records)} "
         f"({'tidak ada perubahan' if not any(r['action'] == 'MODIFIED' for r in cleaning_records) else 'lihat cleaning_log.csv'})",
         f"- Baris dikeluarkan: {len(dropped_ids)} (drop_log.csv)",
         f"- image_integrity_flag=True: {sorted(flag_ids) or '-'}", "",
         "## Demonstrasi fit per fold (tanpa melatih model)", ""]
for name, r in DEMO_RESULTS.items():
    lines.append(f"- **{name}**: train {r['n_train']}, val {r['n_val']}, NaN sesudah transform 0, cek {r['checks']}")
lines += ["", "## Audit akhir", "",
          "- Kolom terlarang di X: tidak ada", "- Target tidak masuk fitur: OK", "- ID hilang/ganda: tidak ada",
          "- Transformer di-fit hanya pada train (statistics_/mean_/scale_/n_samples_seen_ = statistik train): OK",
          f"- Kebocoran citra identik antar-fold ({', '.join(SPLIT_COLS)}): tidak ada", "",
          "## Perlu ditinjau", ""] + ([f"- {t}" for t in REVIEW] or ["- (tidak ada)"])
SUMMARY_PATH.write_text("\n".join(lines) + "\n", encoding="utf-8")
print("Tersimpan:", SUMMARY_PATH)

### CELL 8b — Verifikasi akhir

In [ ]:
assert sha256_file(CLEANED_PATH) == HASH_MASTER_BEFORE and sha256_file(FOLDS_PATH) == HASH_FOLDS_BEFORE
outputs = [PHENO_CLEAN_PATH, CLEANING_LOG_PATH, DROP_LOG_PATH, MISSING_DECISION_PATH, SUMMARY_PATH, PIPELINE_MODULE_PATH]
for p in outputs:
    assert p.exists() and (p.stat().st_size > 0), f"Output hilang/kosong: {p}"
print("=== RINGKASAN 05b ===")
print(f"Subjek          : {len(data)} | ASD {int(y.sum())} | TD {int((y == 0).sum())} | dev {int(IS_DEV.sum())} | "
      f"test {int(IS_TEST.sum())} | eksklusi 05a {len(EXCL_IDS)} | dikeluarkan 05b {len(dropped_ids)}")
print(f"Fitur terpilih  : {SELECTED_FEATURES}")
print(f"Fitur akhir     : tree {len(FINAL_FEATURES['tree'])} | linear_nn {len(FINAL_FEATURES['linear_nn'])}")
print(f"NaN sesudah transform: 0 di {len(DEMO_RESULTS)} demonstrasi")
print(f"Hash master     : {'tidak berubah' if HASH_MASTER_AFTER == HASH_MASTER_BEFORE else 'BERUBAH'}")
print(f"Hash folds      : {'tidak berubah' if HASH_FOLDS_AFTER == HASH_FOLDS_BEFORE else 'BERUBAH'}")
print(f"Perlu ditinjau  : {len(REVIEW)}")
for t in REVIEW:
    print("  -", t)
print("Output:", [p.relative_to(PROJECT_ROOT).as_posix() for p in outputs])